### 加载数据

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
#from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")

In [2]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)
# Inf值检查
inf_cols = X.columns[np.isinf(X).any(axis=0)]
print(inf_cols.tolist())
# 异常收益检查
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

[]
数据异常：收益率超过±30%的列已剔除 3 列 -> ['161811.SZ', '510030.SH', '511580.SH']


### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

In [5]:
from Pre_selection import DropTailCorrelated
from skfolio import Population,MultiPeriodPortfolio
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure
from sklearn.pipeline import Pipeline
from skfolio.optimization import EqualWeighted
from skfolio.pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated
from skfolio.model_selection import WalkForward

In [13]:
selection_pipe = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated(min_n_assets=10,
                        fitness_measures=[PerfMeasure.MEAN, RiskMeasure.VARIANCE,
                                           RatioMeasure.SHARPE_RATIO])),
        ("correlate", DropCorrelated(threshold=0.2, absolute=False)),
    ])

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
    - purged_size=0：训练结束与测试开始无缝衔接。
    - purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。
建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。

- 训练集扩展与尾部数据处理
    - expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
    - reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。

In [15]:
train_portfolios = []
test_portfolios = []

cv = WalkForward(test_size=252, train_size=int(252*3), purged_size=1, reduce_test=True, expand_train=False)
for i, (train_index, test_index) in enumerate(cv.split(X)):
    # 划分训练测试集
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    # 完整性筛选
    X_train = selection_pipe.fit_transform(X_train)
    if X_train.empty:
        continue
    # 训练模型
    m = EqualWeighted(portfolio_params=dict(name="Fold %d"%i)).fit(X_train)
    train_portfolios.append(m.predict(X_train))
    test_portfolios.append(m.predict(X_test[X_train.columns]))

population_train = Population(train_portfolios)
population_test = Population(MultiPeriodPortfolio(test_portfolios))

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population = population_train + population_test

In [17]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [24]:
population_test.plot_cumulative_returns()

In [23]:
MultiPeriodPortfolio(test_portfolios).plot_cumulative_returns()